In [ ]:
%gherkin
Feature: Mask invoice_number in the d_product_revenue_clone table

  Background:
    Given a Unity Catalog at purgo_playground
    And a source table purgo_playground.d_product_revenue exists
    And a clone table purgo_playground.d_product_revenue_clone does not exist or is dropped

  Scenario: Clone d_product_revenue table
    Given the original table purgo_playground.d_product_revenue
    When I drop the table purgo_playground.d_product_revenue_clone if it exists
    Then I create a replica of the original table as purgo_playground.d_product_revenue_clone
    
  Scenario Outline: Mask invoice_number with '*' in the clone table
    Given the cloned table purgo_playground.d_product_revenue_clone
    When I retrieve invoice_number as <original_invoice_number>
    Then I mask the last 4 digits resulting in <masked_invoice_number>
    
    Examples:
      | original_invoice_number | masked_invoice_number |
      | 1234234534              | 123423****            |
      | 9876543210              | 987654****            |
      | 1122334455              | 112233****            |

  Scenario: Verify masking of invoice_number in clone table
    Given the purgo_playground.d_product_revenue_clone table
    When the masking process is applied
    Then invoice_number should have the last 4 digits replaced by '*'
    And all masked values should match the expected format

  Scenario: Handle error scenarios for invoice_number data type
    Given the invoice_number column in purgo_playground.d_product_revenue is of type bigint
    When values shorter than 4 digits are encountered
    Then raise an error
    And display "Error: Invoice number less than expected length"

  Scenario: Validate access permissions for table operations
    Given Unity Catalog privileges
    When performing drop and clone operations
    Then ensure the user has required permissions
    And confirm environment setup for PySpark execution
